In [ ]:
import csv
import json
import hashlib
import requests
import datetime
#from humanfriendly import format_timespan
import pandas as pd
#from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
import json
#from pymongo import MongoClient
#import pymysql
from datetime import datetime,timedelta
#import matplotlib.pyplot as plt
import ast
import psycopg2


In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 5000)
pd.set_option('display.max_columns', 5000)
pd.set_option('display.width', 100000)



In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py



In [ ]:
pan_data=pd.read_csv('batch_23_pan_ocr-solved.csv')

In [ ]:
adhaar_data=pd.read_csv('batch_23_aadhaar_ocr-solved.csv')

## if file name is bank_user_id

In [ ]:
pan_data=pan_data.rename(columns={'ocr_fields_0_pan_number_value':'pan_number','ocr_fields_0_full_name_value':'pan_name','ocr_fields_0_father_name_value':'pan_father_name','ocr_fields_0_dob_value':'pan_date_of_birth','file_name':'bank_user_id'})    
pan_data=pan_data[['pan_number','pan_name','pan_father_name','pan_date_of_birth','bank_user_id']]
pan_data['bank_user_id']=pan_data.bank_user_id.apply(lambda x : x[:-9])
pan_data['pan_number']=pan_data.pan_number.fillna('0')
pan_data['pan_length']=pan_data.pan_number.apply(len)


adhaar_data=adhaar_data[['ocr_fields_0_aadhaar_number_value','ocr_fields_1_aadhaar_number_value','ocr_fields_0_dob_value','ocr_fields_1_address_state','ocr_fields_1_zip_value','ocr_fields_1_document_type','ocr_fields_0_full_name_value','ocr_fields_1_address_value','file_name']]  
adhaar_data['ocr_fields_0_aadhaar_number_value']=adhaar_data.fillna('None').apply(lambda x: x['ocr_fields_1_aadhaar_number_value'] if x['ocr_fields_0_aadhaar_number_value']=='None' else x['ocr_fields_0_aadhaar_number_value'],axis=1)  
adhaar_data=adhaar_data.rename(columns={'ocr_fields_0_aadhaar_number_value':'adhaar_number','ocr_fields_0_dob_value':'adhaar_dob','ocr_fields_1_address_value':'adhaar_address','ocr_fields_0_full_name_value':'adhaar_name','ocr_fields_1_address_state':'adhaar_adress_state','ocr_fields_1_zip_value':'adhaar_adress_pincode','file_name':'bank_user_id'})      
adhaar_data['bank_user_id']=adhaar_data['bank_user_id'].apply(lambda x : x[:-19])
adhaar_data=adhaar_data[['adhaar_number','adhaar_name','adhaar_dob','adhaar_address','adhaar_adress_state','adhaar_adress_pincode','bank_user_id']] 

adhaar_data['adhaar_number']=adhaar_data.adhaar_number.fillna('0')
adhaar_data['adhaar_address']=adhaar_data.adhaar_address.fillna('None')
adhaar_data['adhaar_length']=adhaar_data.adhaar_number.apply(len)



## if file name is platform user code

pan_data=pan_data.rename(columns={'ocr_fields_0_pan_number_value':'pan_number','ocr_fields_0_full_name_value':'pan_name','ocr_fields_0_father_name_value':'pan_father_name','ocr_fields_0_dob_value':'pan_date_of_birth','file_name':'user_code'})    
pan_data=pan_data[['pan_number','pan_name','pan_father_name','pan_date_of_birth','user_code']]
pan_data['user_code']=pan_data.user_code.apply(lambda x : x[:-9])
pan_data['pan_number']=pan_data.pan_number.fillna('0')
pan_data['pan_length']=pan_data.pan_number.apply(len)



adhaar_data=adhaar_data[['ocr_fields_0_aadhaar_number_value','ocr_fields_1_aadhaar_number_value','ocr_fields_0_dob_value','ocr_fields_1_address_state','ocr_fields_1_zip_value','ocr_fields_1_document_type','ocr_fields_0_full_name_value','ocr_fields_1_address_value','file_name']]  
adhaar_data['ocr_fields_0_aadhaar_number_value']=adhaar_data.fillna('None').apply(lambda x: x['ocr_fields_1_aadhaar_number_value'] if x['ocr_fields_0_aadhaar_number_value']=='None' else x['ocr_fields_0_aadhaar_number_value'],axis=1)  
adhaar_data=adhaar_data.rename(columns={'ocr_fields_0_aadhaar_number_value':'adhaar_number','ocr_fields_0_dob_value':'adhaar_dob','ocr_fields_1_address_value':'adhaar_address','ocr_fields_0_full_name_value':'adhaar_name','ocr_fields_1_address_state':'adhaar_adress_state','ocr_fields_1_zip_value':'adhaar_adress_pincode','file_name':'user_code'})      
adhaar_data['user_code']=adhaar_data['user_code'].apply(lambda x : x[:-12])
adhaar_data=adhaar_data[['adhaar_number','adhaar_name','adhaar_dob','adhaar_address','adhaar_adress_state','adhaar_adress_pincode','user_code']] 
adhaar_data['adhaar_number']=adhaar_data.adhaar_number.fillna('0')
adhaar_data['adhaar_address']=adhaar_data.adhaar_address.fillna('None')
adhaar_data['adhaar_length']=adhaar_data.adhaar_number.apply(len)

user_code_bank_user_id=pd.read_sql("select user_code,bank_user_id from bank_users where user_code in {} and bank_name='PARTNER_BANK' ".format(tuple(adhaar_data['user_code'])),con) 
adhaar_data=pd.merge(adhaar_data,user_code_bank_user_id)

In [ ]:
overall_data

In [ ]:
overall_data=pd.merge(adhaar_data,pan_data)

### data to be added to the 26k sheet

In [ ]:
overall_data[(overall_data['adhaar_length']<12)|(overall_data['pan_length']<10)|(overall_data['adhaar_address']=='None')][['user_code','adhaar_name','adhaar_number','adhaar_address','pan_name','pan_number']].to_csv('additional_data_for_26k.csv')  



### data ready to send to bank


In [ ]:
overall_data[(overall_data['adhaar_length']==12)&(overall_data['pan_length']==10)&(overall_data['adhaar_address']!='None')].to_csv('set_44.csv')